# 🛍️ Shopping Mall Customer Segmentation
## Member 1 — K-Means Clustering
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  
**Algorithm:** K-Means Clustering  
**Task:** Unsupervised Machine Learning — Customer Segmentation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load Pre-processed Data
> Data was already cleaned and scaled in `step0_eda_preprocessing.ipynb`.


In [ ]:
# Load preprocessed data from step0_eda_preprocessing.ipynb output
# ⚠️ Make sure you have run step0_eda_preprocessing.ipynb first!
import numpy as np
import pandas as pd

X_scaled = np.load('X_scaled.npy')
df_clean  = pd.read_csv('data_preprocessed.csv')
df        = pd.read_csv('Shopping_Mall_Customer_Segmentation_Data_.csv')

print('✅ Loaded X_scaled.npy   shape:', X_scaled.shape)
print('✅ Loaded data_preprocessed.csv shape:', df_clean.shape)
print('✅ Loaded original CSV   shape:', df.shape)

## 5. Find Optimal K — Elbow Method & Silhouette Score

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    print(f'K={k} | Inertia: {km.inertia_:.2f} | Silhouette: {silhouette_score(X_scaled, labels):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — Finding Optimal K')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=150)
plt.show()

## 6. Train Final K-Means Model

In [ ]:
# Choose optimal K based on elbow + silhouette (adjust if needed)
OPTIMAL_K = silhouette_scores.index(max(silhouette_scores)) + 2
print(f'Optimal K selected: {OPTIMAL_K}')

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df_clean['Cluster'] = cluster_labels
df['Cluster'] = cluster_labels

print('\nCluster Distribution:')
print(df['Cluster'].value_counts().sort_index())

## 7. Evaluation Metrics

In [ ]:
sil = silhouette_score(X_scaled, cluster_labels)
dbi = davies_bouldin_score(X_scaled, cluster_labels)
inertia = kmeans.inertia_

print('=' * 45)
print('       K-MEANS EVALUATION METRICS')
print('=' * 45)
print(f'  Number of Clusters  : {OPTIMAL_K}')
print(f'  Inertia (WCSS)      : {inertia:.4f}')
print(f'  Silhouette Score    : {sil:.4f}  (closer to 1 = better)')
print(f'  Davies-Bouldin Index: {dbi:.4f}  (closer to 0 = better)')
print('=' * 45)

## 8. Cluster Visualization (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained Variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.2f}%')

plt.figure(figsize=(9, 6))
palette = sns.color_palette('tab10', OPTIMAL_K)
for c in range(OPTIMAL_K):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=palette[c], label=f'Cluster {c}', alpha=0.5, s=15)

# Plot centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            marker='X', s=200, c='black', zorder=5, label='Centroids')

plt.title(f'K-Means Clustering (K={OPTIMAL_K}) — PCA 2D View')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('kmeans_pca_clusters.png', dpi=150)
plt.show()

## 9. Cluster Profile Analysis

In [ ]:
profile = df.groupby('Cluster')[['Age', 'Annual Income', 'Spending Score']].mean().round(2)
profile['Count'] = df.groupby('Cluster')['Age'].count()
profile['% of Total'] = (profile['Count'] / len(df) * 100).round(2)
print('Cluster Profiles:')
profile

In [ ]:
# Heatmap of cluster profiles
plt.figure(figsize=(8, 4))
heatmap_data = df.groupby('Cluster')[['Age', 'Annual Income', 'Spending Score']].mean()
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('K-Means Cluster Profile Heatmap')
plt.tight_layout()
plt.savefig('kmeans_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Box plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features = ['Age', 'Annual Income', 'Spending Score']
colors_box = sns.color_palette('tab10', OPTIMAL_K)

for i, feat in enumerate(features):
    df.boxplot(column=feat, by='Cluster', ax=axes[i])
    axes[i].set_title(f'{feat} by Cluster')
    axes[i].set_xlabel('Cluster')

plt.suptitle('')
plt.tight_layout()
plt.savefig('kmeans_boxplots.png', dpi=150)
plt.show()

## 10. Summary

In [ ]:
print('=' * 55)
print('         K-MEANS CLUSTERING — FINAL SUMMARY')
print('=' * 55)
print(f'  Algorithm           : K-Means')
print(f'  Optimal K           : {OPTIMAL_K}')
print(f'  Total Customers     : {len(df)}')
print(f'  Inertia (WCSS)      : {inertia:.4f}')
print(f'  Silhouette Score    : {sil:.4f}')
print(f'  Davies-Bouldin Index: {dbi:.4f}')
print('=' * 55)
print('\nCluster Sizes:')
for c, count in df['Cluster'].value_counts().sort_index().items():
    pct = count / len(df) * 100
    print(f'  Cluster {c}: {count} customers ({pct:.1f}%)')